In [1]:
import warnings
# Suppress scikit-learn and LightGBM warnings
warnings.filterwarnings("ignore", category=FutureWarning, message=".*force_all_finite.*")
warnings.filterwarnings("ignore", message="X does not have valid feature names")

import numpy as np
import pandas as pd
from pathlib import Path
import os
import sys
import logging
import traceback
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
from sklearn.cluster import KMeans
from google.colab import drive
from typing import Dict, Tuple, List, Any, Optional

# ====================
# 1. MOUNT DRIVE & PATH CONFIGURATION
# ====================

print("="*50)
print("PATH CONFIGURATION")
print("="*50)

# Mount the Drive
drive.mount('/content/drive', force_remount=True)
mydrive_path = Path("/content/drive/MyDrive")

# Add the "Modules" folder path to the Python path list
modules_path = mydrive_path / "PFL" / "Modules"
if str(modules_path) not in sys.path:
    sys.path.append(str(modules_path))
    print(f"✓ Path {modules_path} added to sys.path for PFL import")

# Import the PFL module
try:
    import PFLZ
except ImportError:
    raise ImportError("❌ ERROR: PFL module not found. Check that the file exists at the specified location.")

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# ====================
# 2. DATA LOADING
# ====================

print("\n" + "="*50)
print("LOADING DATA")
print("="*50)

# Search and verify the data folder
modeling_data2_path = next((Path(root) / "modeling_data2" for root, dirs, files in os.walk(mydrive_path) if "modeling_data2" in dirs), None)
if modeling_data2_path is None:
    raise FileNotFoundError("❌ ERROR: modeling_data2 folder not found")
print(f"✓ modeling_data2 folder found: {modeling_data2_path}")

# Load data
data = {}
data_files = {
    'df_train': 'df_train.hdf', 'X_train': 'X_train.hdf', 'y_train': 'y_train.hdf',
    'X_val': 'X_val.hdf', 'y_val': 'y_val.hdf', 'instance_weights': 'instance_weights_train.hdf'
}

for name, file in data_files.items():
    path = modeling_data2_path / file
    if not path.exists():
        raise FileNotFoundError(f"❌ File not found: {file}")
    df = pd.read_hdf(path)
    if name == 'df_train':
        # Read column A here
        df_train_temp = df
        feature_names = df.columns[2:-3].tolist()
        data['A_train'] = df['A'].values
    data[name] = df.values.ravel() if name.startswith('y') or 'weights' in name else df.values

# Features
feature_names = feature_names if 'feature_names' in locals() else [f'feature_{i}' for i in range(data['X_train'].shape[1])]
print("\n✅ Data loaded successfully!")
print(f"Dimensions of X_train: {data['X_train'].shape}")
print(f"Number of features: {len(feature_names)}")

# ====================
# 3. DEFINITION OF PFL MODULE EXTENSIONS
# ====================

# The extension logic is integrated here.
class PFS_CP_Extended(PFLZ.PFLZ):
    def __init__(self, cluster_centers, epsilon=1e-8, lgb_params=None, feature_names=None):
        n_clusters_from_centers = cluster_centers.shape[0] if cluster_centers.shape[0] > 0 else 1
        super().__init__(n_clusters=n_clusters_from_centers, epsilon=epsilon, lgb_params=lgb_params, feature_names=feature_names)
        self.cluster_centers_ = cluster_centers
        self.feature_names = feature_names

    def calc_mf(self, X: np.ndarray, normalized: bool = False) -> np.ndarray:
        if not hasattr(self, 'rule_widths_') or self.rule_widths_ is None:
            self.rule_widths_ = super().calc_rule_widths()
        diff = X[:, np.newaxis, :] - self.cluster_centers_
        sq_distances = np.sum(diff ** 2, axis=2)
        min_dists = self.rule_widths_[:, 0] if self.rule_widths_.shape[0] > 0 else np.array([1.0])
        sigma = np.median(min_dists)
        gamma = 1.0 / (2 * sigma ** 2 + self.epsilon)
        mf_values = np.clip(np.exp(-gamma * sq_distances), 0, 1)
        if normalized:
            row_sum = np.sum(mf_values, axis=1, keepdims=True)
            row_sum = np.where(row_sum < self.epsilon, self.epsilon, row_sum)
            mf_normalized = mf_values / row_sum
            if np.isnan(mf_normalized).any():
                uniform_vals = np.full(mf_normalized.shape, 1.0 / mf_normalized.shape[1])
                mf_normalized = np.where(np.isnan(mf_normalized), uniform_vals, mf_normalized)
            return mf_normalized
        return mf_values

    def fit(self, X, y):
        super().fit(X, y)
        self.X_train_ = X
        self.y_train_ = y
        self.n_original_features_ = X.shape[1]
        return self

    def get_feature_importances(self):
        if not hasattr(self, 'X_train_') or self.lgb_model is None:
            raise ValueError("Data or model not trained.")
        fuzzy_features = self.calc_mf(self.X_train_, normalized=True)
        feature_names_extended = self.feature_names + [f'fuzzy_feature_{i}' for i in range(fuzzy_features.shape[1])]
        X_extended = pd.DataFrame(np.concatenate((self.X_train_, fuzzy_features), axis=1), columns=feature_names_extended)
        baseline_preds = self.lgb_model.predict_proba(X_extended)[:, 1]
        if len(np.unique(self.y_train_)) < 2:
            self.bayesian_importances_ = np.zeros(self.n_original_features_)
            return
        baseline_score = roc_auc_score(self.y_train_, baseline_preds)
        bayesian_importances = []
        for i in range(self.n_original_features_):
            X_permuted_df = pd.DataFrame(self.X_train_.copy(), columns=self.feature_names)
            np.random.shuffle(X_permuted_df.iloc[:, i].values)
            fuzzy_features_perm = self.calc_mf(X_permuted_df.values, normalized=True)
            X_extended_perm = pd.DataFrame(np.concatenate((X_permuted_df.values, fuzzy_features_perm), axis=1), columns=feature_names_extended)
            preds_perm = self.lgb_model.predict_proba(X_extended_perm)[:, 1]
            score_perm = roc_auc_score(self.y_train_, preds_perm)
            delta = baseline_score - score_perm
            prob_relevance = 1.0 / (1.0 + np.exp(-50 * delta))
            bayesian_importances.append(prob_relevance)
        self.bayesian_importances_ = np.array(bayesian_importances)
        feature_importances = sorted([(self.bayesian_importances_[i], self.feature_names[i]) for i in range(self.n_original_features_)], reverse=True)
        print("\nBayesian feature selection and ranking:")
        for imp, feat_name in feature_importances:
            print(f"   {feat_name}: relevance probability = {imp:.4f}")

    def select_top_features(self, top_k=7):
        if not hasattr(self, 'bayesian_importances_'):
            raise ValueError("Bayesian importances have not been calculated.")
        top_indices = np.argsort(self.bayesian_importances_)[::-1][:top_k]
        selected_features = [self.feature_names[i] for i in top_indices]
        self.selected_features_ = selected_features
        return selected_features

def maximize_pfs_cp_final(X_train, y_train, X_val, y_val, sample_weight, n_rules=4, iteration=42, feature_names=None):
    km = KMeans(n_clusters=n_rules, random_state=iteration, n_init='auto')
    km.fit(X_train, sample_weight=sample_weight)
    best_lgb_params = {'n_estimators': 200, 'learning_rate': 0.05, 'num_leaves': 63, 'verbose': -1, 'random_state': 42, 'objective': 'binary', 'metric': 'auc'}
    pfs = PFS_CP_Extended(cluster_centers=km.cluster_centers_, lgb_params=best_lgb_params, feature_names=feature_names)
    pfs.fit(X_train, y_train)
    pfs.get_feature_importances()
    selected_features = pfs.select_top_features(top_k=7)
    print("\nFinal selection (top 7 features):")
    for feat in selected_features:
        print(f"    {feat}")

    preds = pfs.predict_proba(X_val)[:, 1]
    roc_auc = roc_auc_score(y_val, preds)
    print(f"\n[PFS_CP Maximization] ROC AUC = {roc_auc:.4f}")
    return roc_auc, preds, pfs

# Define functions for rule extraction and display
def extract_if_then_rules(model, all_feature_names, n_features=7, n_rules=4, alpha=1.0):
    rules = []
    effective_n_clusters = model.cluster_centers_.shape[0]
    n_to_use = min(n_rules, effective_n_clusters)
    selected_indices = [all_feature_names.index(f) for f in model.selected_features_]
    centers = model.cluster_centers_[:, selected_indices][:n_to_use, :n_features]
    widths = model.calc_rule_widths()[:, selected_indices][:n_to_use, :n_features] if hasattr(model, 'calc_rule_widths') else np.ones_like(centers) * 2.0
    cond_probs = model.cond_probs_[:n_to_use, :]
    for i in range(n_to_use):
        rule_conditions = [f"{model.selected_features_[j]} ∈ [{centers[i, j] - alpha * widths[i, j]:.2f}, {centers[i, j] + alpha * widths[i, j]:.2f}]" for j in range(min(n_features, len(model.selected_features_)))]
        pred_class = np.argmax(cond_probs[i])
        prob = cond_probs[i, pred_class]
        rule_text = "IF " + " AND ".join(rule_conditions) + f" THEN class = {pred_class} (probability: {prob:.3f})"
        rules.append(rule_text)
    return rules

def print_rules(rules):
    print("\n\nExtracted 'if-then' rules:")
    for idx, rule in enumerate(rules, start=1):
        print(f"\nRule {idx}:\n    {rule}")

# ====================
# 4. MODEL EXECUTION PER SUBSET
# ====================

print("\n" + "="*50)
print("MODEL TRAINING AND EVALUATION PER SUBSET")
print("="*50)

# Prepare data subsets based on the 'A' variable
mask_A0 = data['A_train'] == 0
mask_A1 = data['A_train'] == 1

X_train_no_A = data['X_train']
y_train_no_A = data['y_train']
weights_no_A = data['instance_weights']

subsets = {
    'A=0': (X_train_no_A[mask_A0], y_train_no_A[mask_A0], weights_no_A[mask_A0]),
    'A=1': (X_train_no_A[mask_A1], y_train_no_A[mask_A1], weights_no_A[mask_A1]),
    'ALL': (X_train_no_A, y_train_no_A, weights_no_A)
}

# Execute the model for each subset
for subset_name, (X_train_subset, y_train_subset, weights_subset) in subsets.items():
    print(f"\n{'='*20} Starting processing for subset: {subset_name} {'='*20}")
    try:
        roc_auc, preds, model = maximize_pfs_cp_final(
            X_train=X_train_subset,
            y_train=y_train_subset,
            X_val=data['X_val'], # Keep the global validation set for comparison
            y_val=data['y_val'],
            sample_weight=weights_subset,
            n_rules=4,
            iteration=42,
            feature_names=feature_names
        )
        print(f"\n✅ Model trained successfully for {subset_name}! ROC AUC: {roc_auc:.4f}")

        # Extract and display rules for this subset
        rules = extract_if_then_rules(model, all_feature_names=feature_names)
        print_rules(rules)

        print(f"\n{'='*20} End of processing for subset: {subset_name} {'='*20}\n")
    except Exception as e:
        print(f"\n❌ EXECUTION ERROR for {subset_name}: {e}")
        traceback.print_exc()
        sys.exit(1)

print("\n" + "="*50)
print("✅ EXECUTION COMPLETED SUCCESSFULLY")
print("="*50)

PATH CONFIGURATION
Mounted at /content/drive
✓ Path /content/drive/MyDrive/PFL/Modules added to sys.path for PFL import

LOADING DATA
✓ modeling_data2 folder found: /content/drive/MyDrive/PFL/modeling_data2

✅ Data loaded successfully!
Dimensions of X_train: (672008, 27)
Number of features: 27

MODEL TRAINING AND EVALUATION PER SUBSET

==================== Starting processing for subset: A=0 ====================



Bayesian feature selection and ranking:
   urea: relevance probability = 0.6294
   got(asat): relevance probability = 0.6275
   gpt(alat): relevance probability = 0.6233
   bilirubin: relevance probability = 0.6151
   platelets: relevance probability = 0.5927
   white blood cells: relevance probability = 0.5893
   creatinine: relevance probability = 0.5490
   bicarbonate: relevance probability = 0.5480
   ptt: relevance probability = 0.5457
   fio2: relevance probability = 0.5431
   crp: relevance probability = 0.5379
   arterial pco2: relevance probability = 0.5359
   calcium: relevance probability = 0.5324
   hematocrit: relevance probability = 0.5282
   potassium: relevance probability = 0.5281
   heart rate: relevance probability = 0.5249
   glucose: relevance probability = 0.5227
   arterial po2: relevance probability = 0.5223
   arterial ph: relevance probability = 0.5187
   sodium: relevance probability = 0.5184
   arterial base excess: relevance probability = 0.5153
   spo2: r


[PFS_CP Maximization] ROC AUC = 0.7441

✅ Model trained successfully for A=0! ROC AUC: 0.7441


Extracted 'if-then' rules:

Rule 1:
    IF urea ∈ [-1.55, 1.85] AND got(asat) ∈ [-1.82, 1.58] AND gpt(alat) ∈ [-1.85, 1.56] AND bilirubin ∈ [-1.89, 1.51] AND platelets ∈ [-1.78, 1.63] AND white blood cells ∈ [-1.75, 1.65] AND creatinine ∈ [-1.75, 1.65] THEN class = 0 (probability: 0.920)

Rule 2:
    IF urea ∈ [-5.04, 4.42] AND got(asat) ∈ [-4.90, 4.56] AND gpt(alat) ∈ [-4.95, 4.51] AND bilirubin ∈ [-5.16, 4.30] AND platelets ∈ [-3.37, 6.09] AND white blood cells ∈ [-5.21, 4.25] AND creatinine ∈ [-5.33, 4.12] THEN class = 0 (probability: 0.902)

Rule 3:
    IF urea ∈ [-2.42, 2.85] AND got(asat) ∈ [-1.90, 3.38] AND gpt(alat) ∈ [-1.61, 3.67] AND bilirubin ∈ [-1.41, 3.87] AND platelets ∈ [-3.13, 2.15] AND white blood cells ∈ [-2.08, 3.20] AND creatinine ∈ [-2.37, 2.91] THEN class = 0 (probability: 0.871)

Rule 4:
    IF urea ∈ [-2.13, 1.27] AND got(asat) ∈ [-1.85, 1.55] AND gpt(alat) ∈ [-1.90,


Bayesian feature selection and ranking:
   got(asat): relevance probability = 0.6496
   platelets: relevance probability = 0.5983
   gpt(alat): relevance probability = 0.5859
   bilirubin: relevance probability = 0.5650
   ptt: relevance probability = 0.5395
   calcium: relevance probability = 0.5324
   white blood cells: relevance probability = 0.5314
   creatinine: relevance probability = 0.5250
   fio2: relevance probability = 0.5180
   urea: relevance probability = 0.5153
   crp: relevance probability = 0.5144
   cvp: relevance probability = 0.5104
   sodium: relevance probability = 0.5095
   potassium: relevance probability = 0.5072
   glucose: relevance probability = 0.5063
   heart rate: relevance probability = 0.5058
   arterial po2: relevance probability = 0.5056
   arterial ph: relevance probability = 0.5054
   hematocrit: relevance probability = 0.5054
   bicarbonate: relevance probability = 0.5047
   arterial base excess: relevance probability = 0.5030
   temperature: rele


[PFS_CP Maximization] ROC AUC = 0.7985

✅ Model trained successfully for A=1! ROC AUC: 0.7985


Extracted 'if-then' rules:

Rule 1:
    IF got(asat) ∈ [-1.52, 1.29] AND platelets ∈ [-1.42, 1.39] AND gpt(alat) ∈ [-1.54, 1.27] AND bilirubin ∈ [-1.54, 1.27] AND ptt ∈ [-1.53, 1.28] AND calcium ∈ [-1.62, 1.19] AND white blood cells ∈ [-1.48, 1.33] THEN class = 0 (probability: 0.921)

Rule 2:
    IF got(asat) ∈ [-1.55, 1.26] AND platelets ∈ [-1.08, 1.73] AND gpt(alat) ∈ [-1.59, 1.21] AND bilirubin ∈ [-1.72, 1.09] AND ptt ∈ [-1.25, 1.56] AND calcium ∈ [-1.60, 1.20] AND white blood cells ∈ [-1.43, 1.38] THEN class = 0 (probability: 0.929)

Rule 3:
    IF got(asat) ∈ [-2.06, 7.47] AND platelets ∈ [-5.17, 4.36] AND gpt(alat) ∈ [-2.07, 7.46] AND bilirubin ∈ [-4.19, 5.33] AND ptt ∈ [-4.22, 5.30] AND calcium ∈ [-4.99, 4.53] AND white blood cells ∈ [-4.16, 5.37] THEN class = 0 (probability: 0.680)

Rule 4:
    IF got(asat) ∈ [-2.55, 2.73] AND platelets ∈ [-3.07, 2.21] AND gpt(alat) ∈ [-2.49, 2.79] 


Bayesian feature selection and ranking:
   gpt(alat): relevance probability = 0.7933
   got(asat): relevance probability = 0.7292
   platelets: relevance probability = 0.6856
   urea: relevance probability = 0.6816
   bilirubin: relevance probability = 0.6472
   white blood cells: relevance probability = 0.6396
   ptt: relevance probability = 0.6018
   creatinine: relevance probability = 0.5857
   calcium: relevance probability = 0.5695
   potassium: relevance probability = 0.5652
   fio2: relevance probability = 0.5650
   A: relevance probability = 0.5561
   crp: relevance probability = 0.5557
   arterial pco2: relevance probability = 0.5515
   hematocrit: relevance probability = 0.5508
   heart rate: relevance probability = 0.5446
   bicarbonate: relevance probability = 0.5433
   sodium: relevance probability = 0.5422
   arterial ph: relevance probability = 0.5397
   arterial po2: relevance probability = 0.5350
   cvp: relevance probability = 0.5349
   arterial base excess: relevanc


[PFS_CP Maximization] ROC AUC = 0.7753

✅ Model trained successfully for ALL! ROC AUC: 0.7753


Extracted 'if-then' rules:

Rule 1:
    IF gpt(alat) ∈ [-1.99, 6.40] AND got(asat) ∈ [-2.30, 6.08] AND platelets ∈ [-4.49, 3.90] AND urea ∈ [-3.99, 4.39] AND bilirubin ∈ [-3.91, 4.47] AND white blood cells ∈ [-3.50, 4.89] AND ptt ∈ [-3.71, 4.68] THEN class = 0 (probability: 0.806)

Rule 2:
    IF gpt(alat) ∈ [-2.25, 1.97] AND got(asat) ∈ [-2.22, 2.00] AND platelets ∈ [-2.23, 1.99] AND urea ∈ [-1.95, 2.27] AND bilirubin ∈ [-2.30, 1.92] AND white blood cells ∈ [-2.16, 2.06] AND ptt ∈ [-2.06, 2.16] THEN class = 0 (probability: 0.919)

Rule 3:
    IF gpt(alat) ∈ [-1.99, 2.23] AND got(asat) ∈ [-2.06, 2.16] AND platelets ∈ [-2.57, 1.65] AND urea ∈ [-2.02, 2.20] AND bilirubin ∈ [-0.52, 3.70] AND white blood cells ∈ [-1.80, 2.42] AND ptt ∈ [-1.86, 2.36] THEN class = 0 (probability: 0.900)

Rule 4:
    IF gpt(alat) ∈ [-2.58, 2.16] AND got(asat) ∈ [-2.53, 2.21] AND platelets ∈ [-1.75, 2.99] AND urea 